# Stage1 grouped synthetic transfer-learning trial
Free T4, bounded to under one hour. No competition submission.

In [ ]:
from pathlib import Path
import json,hashlib,zipfile,tempfile,subprocess,sys,time
STARTED=time.monotonic()
WORK=Path(tempfile.mkdtemp(prefix='stage1-trial-',dir='/kaggle/working'))
items=list(Path('/kaggle/input').rglob('stage1-trial-assets.json'))
if len(items)!=1:raise ValueError('Expected exactly one Stage1 trial asset manifest')
asset=items[0].parent
for name,digest in json.loads(items[0].read_text()).items():
    if hashlib.sha256((asset/name).read_bytes()).hexdigest()!=digest:raise ValueError('Bundle checksum mismatch')
with zipfile.ZipFile(asset/'stage1-trial.bin') as z:
    for name in z.namelist():
        if not (WORK/name).resolve().is_relative_to(WORK.resolve()):raise ValueError('ZIP path escape')
    z.extractall(WORK)
VENV=Path(tempfile.mkdtemp(prefix='stage1-pinned-',dir='/tmp'))
subprocess.run([sys.executable,'-m','venv','--without-pip',str(VENV)],check=True)
PYTHON=str(VENV/'bin/python')
started=time.monotonic()
with (WORK/'install.log').open('w') as log:
    installed=subprocess.run([sys.executable,'-m','pip','--python',PYTHON,'install','--no-cache-dir','-r',str(WORK/'requirements.txt')],stdout=log,stderr=subprocess.STDOUT,timeout=600)
install=dict(exit_code=installed.returncode,seconds=time.monotonic()-started)
(WORK/'install.json').write_text(json.dumps(install,indent=2))
if installed.returncode:raise RuntimeError('Installation failed')
remaining=min(2500,int(3200-(time.monotonic()-STARTED)))
if remaining<120:raise RuntimeError('Insufficient experiment budget')
with (WORK/'training.log').open('w') as log:
    run=subprocess.run([PYTHON,'-u','src/train_stage1_trial.py','--dataset-dir','dataset','--output-dir','training','--epochs','6','--max-seconds',str(remaining-120)],cwd=WORK,stdout=log,stderr=subprocess.STDOUT,timeout=remaining)
print((WORK/'training.log').read_text()[-12000:])
with zipfile.ZipFile(WORK/'stage1-trial-result.zip','w',zipfile.ZIP_DEFLATED) as z:
    for name in ['training/best.pt','training/report.json','training/validation_predictions.csv','training.log','install.json']:
        if (WORK/name).exists():z.write(WORK/name,name)
if run.returncode:raise RuntimeError('Stage1 trial failed')
print('Runner seconds',time.monotonic()-STARTED)
